In [1]:
# Starting code for turtlebot
import rclpy
if not rclpy.ok():
    rclpy.init()

# creating a function
from robot_wrapper import Robot
# Instantiate Wrapper. The subscribers and publishes the relevant topics and connects the actions.
robot = Robot()

In [2]:
import math
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import LaserScan
from geometry_msgs.msg import Twist

In [6]:
class LidarObserver(Node):
    def __init__(self):
        super().__init__('lidar_observer')
        self.subscription = self.create_subscription(
            LaserScan,
            '/scan',
            self.avoiding_obstacles,
            10
        )
        self.publisher = self.create_publisher(Twist, '/cmd_vel', 10)
        self.safety_latch_engaged = False

    def avoiding_obstacles(self, msg):
        ranges = msg.ranges
        if not ranges:
            return

        twist = Twist()

        # 1. Permanent Safety Latch Check
        if self.safety_latch_engaged:
            twist.linear.x = 0.0
            twist.angular.z = 0.0
            self.publisher.publish(twist)
            return

        # 2. Read direct front distance
        front_distance = ranges[0]
        
        # PRINT THE DATA: This will show up in your Jupyter notebook logs!
        self.get_logger().info(f"LiDAR Points: {len(ranges)} | Front Distance: {front_distance:.2f}m")

        # 3. Stop Logic (Set to 0.50m)
        # We ensure the reading is a valid number, greater than 0, and under 0.50m
        if 0.0 < front_distance <= 0.50 and not math.isinf(front_distance) and not math.isnan(front_distance):
            self.get_logger().warn(f"🚨 TARGET DETECTED AT {front_distance:.2f}m! ENGAGING SAFETY LATCH! 🚨")
            self.safety_latch_engaged = True
            
            twist.linear.x = 0.0
            twist.angular.z = 0.0
            self.publisher.publish(twist)
            return 

        # 4. Clear Path: Move forward very slowly (0.05 m/s) so you can track the logs safely
        twist.linear.x = 0.15  
        twist.angular.z = 0.0
        self.publisher.publish(twist)

In [7]:
def main(args=None):
    if not rclpy.ok():
        rclpy.init(args=args)
    else:
        rclpy.shutdown()
        rclpy.init(args=args)

    node = LidarObserver()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        print("Shutting down node...")
    finally:
        node.destroy_node()
        rclpy.shutdown()

In [ ]:
if __name__ == '__main__':
    main()

[INFO] [1784732587.788274162] [lidar_observer]: LiDAR Points: 720 | Front Distance: infm
[INFO] [1784732587.918879615] [lidar_observer]: LiDAR Points: 720 | Front Distance: infm
[INFO] [1784732588.021799177] [lidar_observer]: LiDAR Points: 720 | Front Distance: 4.54m
[INFO] [1784732588.166957652] [lidar_observer]: LiDAR Points: 720 | Front Distance: 0.75m
[INFO] [1784732588.288366301] [lidar_observer]: LiDAR Points: 720 | Front Distance: 0.75m
[INFO] [1784732588.413329313] [lidar_observer]: LiDAR Points: 720 | Front Distance: infm
[INFO] [1784732588.557000373] [lidar_observer]: LiDAR Points: 720 | Front Distance: 4.54m
[INFO] [1784732588.684689940] [lidar_observer]: LiDAR Points: 720 | Front Distance: 4.56m
[INFO] [1784732588.817707430] [lidar_observer]: LiDAR Points: 720 | Front Distance: 4.55m
[INFO] [1784732588.969062045] [lidar_observer]: LiDAR Points: 720 | Front Distance: 0.91m
[INFO] [1784732589.090462268] [lidar_observer]: LiDAR Points: 720 | Front Distance: 0.88m
[INFO] [17847